# QLoRA RAGAS-style Evaluation (Custom Implementation)

RAGAS 0.4.3 has Groq compatibility issues that no adapter setting can fix.
This notebook implements the same four metrics directly using Groq API calls.

| Metric | How computed |
|---|---|
| `faithfulness` | Groq extracts claims from answer, verifies each against context |
| `answer_relevancy` | Cosine similarity of answer ↔ question embeddings |
| `context_precision` | Groq judges if each retrieved chunk is useful |
| `context_recall` | Groq judges if chunks cover the ground truth |

**No RAGAS evaluate() call — no compatibility issues.**

## 0 · Install

In [ ]:
%pip install -q groq sentence-transformers scikit-learn

## 1 · Config

In [ ]:
import os
from pathlib import Path

os.environ["GROQ_API_KEY"] = "gsk_..."          # ← your Groq key
PINECONE_KEY               = "YOUR_PINECONE_KEY" # ← your Pinecone key

JUDGE_MODEL    = "llama-3.3-70b-versatile"
BASE_MODEL     = "Qwen/Qwen2.5-Coder-7B"
QLORA_ADAPTER  = "Ivan17Ji/qwen-lora-250"
TOP_K          = 10
RETRIEVER_MODE = "advanced"
QUICK_TEST     = True
QUICK_N        = 10
JUDGE_DELAY    = 1.5  # seconds between Groq judge calls

DATA_DIR = Path(".")
print("✅ Config set")

## 2 · Metric implementations

Each metric calls Groq directly — same logic RAGAS uses internally.

In [ ]:
import time, re, json as _json
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

_groq = Groq(api_key=os.environ["GROQ_API_KEY"])
_sem  = SentenceTransformer(
    "flax-sentence-embeddings/st-codesearch-distilroberta-base"
)


def _judge(prompt: str, expect_json: bool = False):
    """Single Groq call with rate-limit delay."""
    time.sleep(JUDGE_DELAY)
    resp = _groq.chat.completions.create(
        model       = JUDGE_MODEL,
        messages    = [{"role": "user", "content": prompt}],
        max_tokens  = 512,
        temperature = 0,
    )
    return resp.choices[0].message.content.strip()


def faithfulness(answer: str, contexts: list[str]) -> float:
    """
    1. Extract claims from the answer.
    2. Verify each claim against the contexts.
    Score = supported_claims / total_claims.
    """
    ctx = "\n---\n".join(contexts[:5])

    # Step 1: extract claims
    extract_prompt = (
        f"Given the following code answer, list every factual claim or statement "
        f"it makes (one per line). If the answer is pure code with no natural language "
        f"claims, describe what the code does in 1-3 short statements.\n\n"
        f"Answer:\n{answer}\n\nClaims (one per line):"
    )
    claims_raw = _judge(extract_prompt)
    claims = [c.strip().lstrip("•-0123456789. ") for c in claims_raw.splitlines()
              if c.strip()]

    if not claims:
        return float("nan")

    # Step 2: verify each claim
    supported = 0
    for claim in claims:
        verify_prompt = (
            f"Context:\n{ctx}\n\n"
            f"Claim: {claim}\n\n"
            f"Is this claim supported by the context? Answer YES or NO only."
        )
        verdict = _judge(verify_prompt).upper()
        if "YES" in verdict:
            supported += 1

    return supported / len(claims)


def answer_relevancy(question: str, answer: str) -> float:
    """Cosine similarity between question and answer embeddings."""
    e = _sem.encode([question, answer])
    return float(cosine_similarity([e[0]], [e[1]])[0][0])


def context_precision(question: str, contexts: list[str], answer: str) -> float:
    """
    For each retrieved chunk: is it useful for answering the question?
    Score = useful_chunks / total_chunks.
    """
    scores = []
    for ctx in contexts[:5]:
        prompt = (
            f"Question: {question}\n\n"
            f"Retrieved chunk:\n{ctx}\n\n"
            f"Is this chunk useful for answering the question? Answer YES or NO only."
        )
        verdict = _judge(prompt).upper()
        scores.append(1.0 if "YES" in verdict else 0.0)
    return sum(scores) / len(scores) if scores else float("nan")


def context_recall(contexts: list[str], ground_truth: str) -> float:
    """
    Can the ground truth answer be inferred from the retrieved chunks?
    Score = 1.0 (yes) or 0.0 (no).
    """
    ctx = "\n---\n".join(contexts[:5])
    prompt = (
        f"Retrieved context:\n{ctx}\n\n"
        f"Ground truth answer:\n{ground_truth}\n\n"
        f"Can the ground truth answer be derived from the retrieved context? "
        f"Answer YES or NO only."
    )
    verdict = _judge(prompt).upper()
    return 1.0 if "YES" in verdict else 0.0


print("✅ Metric functions ready")
print(f"   judge model: {JUDGE_MODEL}")
print(f"   delay: {JUDGE_DELAY}s per call")

## 3 · Load data

In [ ]:
import json as _json, random

with open(DATA_DIR / "ragas_dataset.jsonl") as f:
    all_records = [_json.loads(l) for l in f]
with open(DATA_DIR / "fastapi_golden_set_splits.json") as f:
    splits = _json.load(f)

test_q       = {x["instruction"] for x in splits["test"]}
test_records = [r for r in all_records if r["question"] in test_q]
random.seed(42)
eval_records = random.sample(test_records, QUICK_N) if QUICK_TEST else test_records

print(f"Test split  : {len(test_records)}")
print(f"Eval records: {len(eval_records)}")

## 4 · Load RAG + QLoRA

In [ ]:
import torch, rag_pipeline, importlib
importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG

rag_qlora = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = BASE_MODEL,
    adapter_path = QLORA_ADAPTER,
    ingest_only  = False,
)
rag_qlora.load_local_corpus(str(DATA_DIR / "local_corpus.json"))

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"GPU free: {free:.1f} GB")

## 5 · Generate answers

In [ ]:
from tqdm import tqdm

SYSTEM_PROMPT = (
    "You are a FastAPI expert. Use the provided code context to answer "
    "the question. Return only valid Python code, no explanation."
)

def qwen_generate(question, contexts, max_new_tokens=512):
    context_block = "\n\n".join(contexts[:5])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"### Context\n{context_block}\n\n"
            f"### Question\n{question}\n\n### Answer"
        )},
    ]
    text = rag_qlora.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = rag_qlora.tokenizer(
        text, return_tensors="pt", truncation=True, max_length=4096
    ).to(rag_qlora.model.device)
    with torch.no_grad():
        out = rag_qlora.model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=rag_qlora.tokenizer.eos_token_id,
        )
    return rag_qlora.tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip() or "# no output"


filled, errors = [], 0
for r in tqdm(eval_records, desc="Generating"):
    try:
        ctx = rag_qlora.retrieve_complex(
            r["question"], top_k=TOP_K, mode=RETRIEVER_MODE
        ) or r["contexts"][:TOP_K]
    except Exception as e:
        ctx = r["contexts"][:TOP_K]

    try:
        ans = qwen_generate(r["question"], ctx)
    except Exception as e:
        print(f"  Gen error: {e}")
        ans = "# generation error"
        errors += 1

    filled.append({
        "question":     r["question"],
        "contexts":     ctx,
        "answer":       ans,
        "ground_truth": r["ground_truth"],
    })

print(f"Generated {len(filled)} answers ({errors} errors)")
print(f"Sample: {filled[0]['answer'][:80]!r}")

## 6 · Score each record

Calls Groq for faithfulness, context_precision, context_recall.  
`answer_relevancy` uses local embeddings — no API call.

In [ ]:
import pandas as pd
from tqdm import tqdm

rows = []
for i, x in enumerate(tqdm(filled, desc="Scoring")):
    row = {"question": x["question"], "model": "qwen_qlora"}

    # answer_relevancy — local, no API call
    row["answer_relevancy"] = answer_relevancy(x["question"], x["answer"])

    # faithfulness — Groq
    try:
        row["faithfulness"] = faithfulness(x["answer"], x["contexts"])
    except Exception as e:
        print(f"  faithfulness error [{i}]: {e}")
        row["faithfulness"] = float("nan")

    # context_precision — Groq
    try:
        row["context_precision"] = context_precision(
            x["question"], x["contexts"], x["answer"]
        )
    except Exception as e:
        print(f"  context_precision error [{i}]: {e}")
        row["context_precision"] = float("nan")

    # context_recall — Groq
    try:
        row["context_recall"] = context_recall(x["contexts"], x["ground_truth"])
    except Exception as e:
        print(f"  context_recall error [{i}]: {e}")
        row["context_recall"] = float("nan")

    rows.append(row)
    if (i + 1) % 3 == 0:
        print(f"  [{i+1}/{len(filled)}] "
              f"faith={row['faithfulness']:.2f}  "
              f"rel={row['answer_relevancy']:.2f}  "
              f"prec={row['context_precision']:.2f}  "
              f"rec={row['context_recall']:.2f}")

df = pd.DataFrame(rows)
print("\nDone.")

## 7 · Results

In [ ]:
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
exist = [c for c in METRIC_COLS if c in df.columns]
summary = df[exist].mean().round(3)

print("=" * 45)
print(f"QLoRA RAGAS Results  (n={len(df)})")
print("-" * 45)
for col, val in summary.items():
    bar = ("█" * int(val * 20)) if pd.notna(val) else ""
    print(f"  {col:25s}: {val:.3f}  {bar}" if pd.notna(val)
          else f"  {col:25s}: NaN")
print("=" * 45)
print()
print(df[exist].to_string())

In [ ]:
import matplotlib.pyplot as plt

clean = [(c, summary[c]) for c in exist if pd.notna(summary[c])]
if clean:
    labels, vals = zip(*clean)
    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.bar(range(len(labels)), vals, color="#7F77DD",
                  width=0.5, edgecolor="white")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([l.replace("_", "\n") for l in labels], fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("QLoRA RAGAS-style evaluation")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f"{val:.3f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(DATA_DIR / "qlora_ragas_scores.png", dpi=150)
    plt.show()
    print("Saved: qlora_ragas_scores.png")

## 8 · Save

In [ ]:
df["model"] = "qwen_qlora"
df.to_csv(DATA_DIR / "model_comparison_results.csv", index=False)
summary.to_frame("qwen_qlora").T.to_csv(
    DATA_DIR / "model_comparison_summary.csv"
)
print("Saved: model_comparison_results.csv")
print("Saved: model_comparison_summary.csv")